# Analysing Human Language using R — *talk* + *text* + *topics* workshop

✋ **NOTE** - You need to create a copy of this notebook before you work through it. This can be done by clicking on "Save a copy in Drive" option in the File menu.

<img src="https://r-talk.org/logo.png" alt="talk logo" width="200"> <img src="https://r-text.org/logo.png" alt="text logo" width="200"> <img src="https://r-topics.org/logo.png" alt="topics logo" width="200">

This notebook sets up **one shared Python environment** used by both the [talk](https://r-talk.org) package (audio: transcription, diarisation, speech embeddings) and the [text](https://r-text.org) package (text: language embeddings and analyses), together with the [topics](https://r-topics.org) package (topic modeling of language) — so you can go from a voice recording all the way to text-based analyses and topic models in a single session.


# 0. Setup — one click

Run the single cell below (click the ▶ play button on its left) and let it finish before continuing. It downloads a pre-built environment and takes roughly **5 minutes**. ☕

When it is done you will see **SETUP COMPLETE** at the bottom of the output, together with a test transcription and a test text embedding.

*Optional:* to use a GPU for faster models, first select **Runtime → Change runtime type → T4 GPU** (do this before running the setup cell, because changing runtime later erases the installation).


In [ ]:
## ════════════════════════════════════════════════════════════════
##  ONE-CLICK SETUP — click the play button on this cell, then wait.
##  Takes roughly ~5 minutes. Progress messages appear below.
## ════════════════════════════════════════════════════════════════
t0 <- Sys.time()

## ── Step 1/5: System tools ───────────────────────────────────────
# ffmpeg is required for transcription (whisper loads audio through the
# ffmpeg binary; it must come from apt, NOT conda). Java is needed by
# the text package (rJava).
cat("\n=== Step 1/5: Installing system tools (ffmpeg, Java) ===\n")
system("apt-get update -qq && apt-get install -y -qq ffmpeg pigz openjdk-11-jdk-headless")

## ── Step 2/5: Miniconda via condacolab ───────────────────────────
cat("\n=== Step 2/5: Installing Miniconda (condacolab) ===\n")
cat(system("pip install condacolab gdown 2>&1 | tail -1", intern = TRUE), sep = "\n")
# A failed earlier attempt leaves a '_conda' bootstrap file behind that makes
# every retry fail with "File exists" - clean it up before installing.
system("rm -f /usr/local/_conda")
cat(system("python -c 'import condacolab; condacolab.install()' 2>&1", intern = TRUE), sep = "\n")
if (!nzchar(Sys.which("conda"))) {
  cat(system("tail -15 /content/condacolab_install.log 2>&1", intern = TRUE), sep = "\n")
  stop(paste(
    "Miniconda did not install ('conda' is not on the PATH; see the",
    "installer output and log above). This is usually a transient network",
    "problem - simply RE-RUN THIS CELL. If it fails repeatedly, use the",
    "fallback notebook:",
    "https://colab.research.google.com/github/theharmonylab/talk/blob/main/notebooks/talk_text_topics_workshop_full_install.ipynb",
    sep = "\n"), call. = FALSE)
}

## ── Step 3/5: Download the pre-built environment ─────────────────
# One archive with everything: the shared talkrpp_condaenv conda
# environment (torch, WhisNemo, WhiSPA + the text-package Python
# stack), the talk/text/topics R packages, and all pre-downloaded
# models (whisper, NeMo diarisation, mxbai text embeddings).
cat("\n=== Step 3/5: Downloading pre-built environment (the long step) ===\n")
file_id <- "1d1n0HxYLOVvH4sptbFvjjE76MTyW5DYu"   # talk_text_topics_aug_2026.tar.gz
gdown_out <- system(paste("gdown", file_id, "-O talk_text_topics_aug_2026.tar.gz 2>&1"),
                    intern = TRUE)
cat(gdown_out, sep = "\n")
if (!file.exists("talk_text_topics_aug_2026.tar.gz")) {
  stop(paste(
    "Download of the pre-built environment failed (see the gdown output above).",
    "Common causes:",
    "  1. The Drive file is not shared as 'Anyone with the link'.",
    "  2. The file's download quota is temporarily exceeded (many people",
    "     downloading at once) - wait a few minutes and re-run this cell,",
    "     or use the fallback notebook, which installs everything live:",
    "     https://colab.research.google.com/github/theharmonylab/talk/blob/main/notebooks/talk_text_topics_workshop_full_install.ipynb",
    sep = "\n"), call. = FALSE)
}
cat("Unpacking…\n")
system("tar -I pigz -xf talk_text_topics_aug_2026.tar.gz -C /")
unlink("talk_text_topics_aug_2026.tar.gz")   # free disk space
.libPaths(c("/content/library", .libPaths()))

## ── Step 4/5: Wire Java into R (rJava, needed by text) ───────────
cat("\n=== Step 4/5: Configuring Java for R ===\n")
# Heap size must be set before the JVM starts (Day 3's topic models need it).
options(java.parameters = "-Xmx5000m")
java_home <- dirname(dirname(system2("readlink", c("-f", Sys.which("javac")), stdout = TRUE)))
Sys.setenv(
  JAVA_HOME       = java_home,
  LD_LIBRARY_PATH = paste(file.path(java_home, "lib/server"),
                          Sys.getenv("LD_LIBRARY_PATH"), sep = ":")
)
system("R CMD javareconf")
dyn.load(file.path(java_home, "lib/server/libjvm.so"))
rjava_ok <- tryCatch({ library(rJava); TRUE }, error = function(e) {
  message("Pre-built rJava incompatible with this R — recompiling from source…")
  FALSE
})
if (!rjava_ok) {
  install.packages("rJava", repos = "https://cloud.r-project.org")
  library(rJava)
}
.jinit()   # 0 means the JVM loaded without errors

## ── Step 5/5: Initialize BOTH packages to the shared environment ─
cat("\n=== Step 5/5: Initializing talk and text (shared environment) ===\n")
library(reticulate)
Sys.setenv(RETICULATE_MINICONDA_PATH = system2("conda", c("info", "--base"), stdout = TRUE))
talk::talkrpp_initialize()
library(talk)
text::textrpp_initialize(condaenv = "talkrpp_condaenv", save_profile = FALSE)
library(text)
library(topics)

# NLTK tokenizer data used by the text package (e.g. textDescriptives())
reticulate::py_run_string(
  "import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)"
)

# Quick proof that everything works (all models are already in the
# archive, so this should only take seconds)
wav <- system.file("extdata/test_short.wav", package = "talk")
transcription <- talkText(wav)
print(transcription)
emb <- textEmbed(transcription$transcription,
                 model = "mixedbread-ai/mxbai-embed-large-v1")
cat("textEmbed dimensions:", dim(emb$texts[[1]]), "\n")

## ── Done ─────────────────────────────────────────────────────────
cat("\n============================================================\n")
cat("  SETUP COMPLETE in",
    round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1),
    "minutes — talk, text and topics are loaded and ready!\n")
cat("============================================================\n")


# Day 1 · From audio and text to numbers

One section per step - run them in order. Nobody needs their own recording or dataset: **PART A** (talk) runs on the example audio that ships inside the talk package, and **PART B** (text) on a single sentence we write here.

Everything each day produces is written to `tutorial_output/`, so any step can be reloaded with `readRDS()` instead of recomputed.


In [ ]:
# Everything this script produces - embeddings, transcripts, fitted models,
# figures - is written to tutorial_output/. Nothing is left only in memory, so
# any step can be reloaded with readRDS() instead of recomputed. That is the
# habit to take home: the slow steps run once.
dir.create("tutorial_output", showWarnings = FALSE)


In [ ]:
# ##############################################################################
# PART A  ·  talk - from a recording to words and to sound
# ##############################################################################
#
# Two things talk gives you that a plain transcription service does not:
#   1. diarisation - who spoke when, not just what was said
#   2. audio embeddings - a numeric representation of HOW something was said
#      (prosody, pace, tone), separate from the words themselves

library(talk)

# The one-click setup cell already initialized the shared Python environment
# (talkrpp_condaenv) for BOTH talk and text - no initialization needed here.
find_talkrpp_env()     # must be TRUE
# list_talkrpp_envs()  # every environment R can see, if the above is FALSE


In [ ]:
# ------------------------------------------------------------------------------
# A1. The example audio that ships with talk
# ------------------------------------------------------------------------------
# The package carries two short recordings under inst/extdata, so everyone has
# something that works even without their own data.
#
#   test_short.wav     ~1 second   - fast, for the transcribe/embed demo
#   test_diarise.wav   ~11 seconds - two speakers, for the diarisation demo

wav_short   <- system.file("extdata", "test_short.wav",   package = "talk")
wav_diarise <- system.file("extdata", "test_diarise.wav", package = "talk")

wav_short
file.exists(wav_short)     # must be TRUE
file.exists(wav_diarise)   # must be TRUE

# To use your own recording instead, just point the path somewhere else:
#   wav_short <- "data/my_recording.wav"
#
# Practical notes on your own audio:
#   - .wav and .mp3 both work; mono is fine
#   - keep workshop examples short (30-60 s) - transcription is the slow step
#   - it must be a recording you have consent to process


In [ ]:
# ------------------------------------------------------------------------------
# A2. Speech to text
# ------------------------------------------------------------------------------
# talkText() returns a tibble with one row per file: file_path + transcription.
# (talkTranscribe() is the same function under a second name.)
#
# The FIRST run downloads the Whisper model - a few hundred MB and some
# patience. Later runs reuse the cached model and are much faster.

transcription <- talkText(
  talk_filepaths = wav_short,
  model          = "openai/whisper-small"
)

transcription
transcription$transcription

saveRDS(transcription, "tutorial_output/day1_transcription.rds")

# --- FALLBACK -----------------------------------------------------------------
# transcription <- readRDS("tutorial_output/day1_transcription_precomputed.rds")
# ------------------------------------------------------------------------------


In [ ]:
# ------------------------------------------------------------------------------
# A3. Audio embeddings - the numeric representation of the sound
# ------------------------------------------------------------------------------
# talkEmbed() returns a one-row tibble of Dim1 ... DimN for the whole file.
# This is the step that separates talk from a transcription service: the output
# carries acoustic information that the words alone do not.
#
# use_decoder = FALSE uses the encoder side of the model - the acoustics.
# use_decoder = TRUE also requires audio_transcriptions, and mixes in what the
# model thinks was said.

audio_embedding <- talkEmbed(
  talk_filepaths = wav_short,
  model          = "openai/whisper-small",
  use_decoder    = FALSE
)

audio_embedding
dim(audio_embedding)   # 1 x number of dimensions

saveRDS(audio_embedding, "tutorial_output/day1_audio_embedding.rds")

# --- FALLBACK -----------------------------------------------------------------
# audio_embedding <- readRDS("tutorial_output/day1_audio_embedding_precomputed.rds")
# ------------------------------------------------------------------------------


In [ ]:
# ------------------------------------------------------------------------------
# A4. Diarisation - who spoke when
# ------------------------------------------------------------------------------
# talkTranscribeDiarise() returns one row per speech segment, with columns:
#   speaker, start_timestamp, end_timestamp, message
#
# Note device = "cpu". On macOS that is already the default, but on Windows and
# Linux the default is "cuda" and the call will fail on a machine without an
# NVIDIA GPU. Set it explicitly and it works everywhere.
#
# This is the slowest step of Day 1: model_name = "medium.en" is a large
# English-only download. Start it, then keep reading.

transcript <- talkTranscribeDiarise(
  audio        = wav_diarise,
  num_speakers = 2,
  device       = "cpu",
  verbose      = TRUE
)

transcript

unique(transcript$speaker)   # how many speakers were detected?
transcript$message           # what each of them said

saveRDS(transcript, "tutorial_output/day1_transcript.rds")

# --- FALLBACK -----------------------------------------------------------------
# If the model download failed or transcription is still running when we move
# on, stop it and load the pre-computed transcript. Everything below still works.
#
# transcript <- readRDS("tutorial_output/day1_transcript_precomputed.rds")
# ------------------------------------------------------------------------------


**No restart needed here in Colab.** On a local RStudio installation, talk and text carry *separate* Python environments and only one can be attached per R session - so there you must restart R between PART A and PART B. This notebook uses **one shared environment** for both packages, so you can continue straight on.

In [ ]:
# ##############################################################################
# PART B  ·  text - from written language to word embeddings
# ##############################################################################
#
# The idea: a transformer reads a piece of text and returns a long vector of
# numbers. Texts that mean similar things get similar vectors. Once language is
# numbers, all of your usual statistics apply to it.

library(text)   # already initialized to the shared environment by the setup cell


In [ ]:
# ------------------------------------------------------------------------------
# B1. Which model, and how many layers does it have?
# ------------------------------------------------------------------------------
# textEmbed() has a default, but you should choose deliberately. Three questions:
#
#   Which language(s)?  A monolingual English model will not serve Swedish data.
#                       Multilingual models trade some accuracy for coverage.
#   How big?            Larger models are usually more accurate and much slower.
#                       On a laptop, size is the main thing that will hurt you.
#   Reproducible?       Pin the exact model name in your script and report it in
#                       your paper. "We used BERT" is not reproducible.
#
# Browse models at https://huggingface.co/models

textModels()                                        # already downloaded locally

textModelLayers(target_model = "bert-base-uncased") # 12 hidden layers


In [ ]:
# ------------------------------------------------------------------------------
# B2. Open the box: the hidden states, layer by layer
# ------------------------------------------------------------------------------
# textEmbed() is one convenient call wrapped around three steps. We do them
# separately here, once, so that you can see what a "word embedding" actually is
# before you start trusting one.
#
# Step 1 of 3. textEmbedRawLayers() gives you the raw hidden states: one row per
# TOKEN per LAYER. Nothing has been averaged yet.
#
# We use a single short sentence so this runs in seconds.

example_sentence <- "I am fine"

raw_layers <- textEmbedRawLayers(
  example_sentence,
  model  = "bert-base-uncased",
  layers = 11:12
)

# Find your way around the object rather than trusting my description of it:
names(raw_layers)                        # "context_tokens"
names(raw_layers$context_tokens)         # named after the input; here "texts"

raw_layers$context_tokens$texts[[1]]

# Columns: id, tokens, token_id, layer_number, Dim1 ... Dim768
#
# Three things to notice, and they are the whole lesson:
#
#   1. There is one row per token, not per sentence. "I am fine" became several
#      rows - and note [CLS] and [SEP], the markers BERT adds itself.
#   2. Each token appears TWICE, once for layer 11 and once for layer 12. A
#      transformer does not have "an" embedding; it has one per layer.
#   3. The same word in a different sentence would get different numbers. That
#      is what "contextual" means, and it is the difference from a word list.

table(raw_layers$context_tokens$texts[[1]]$layer_number)

saveRDS(raw_layers, "tutorial_output/day1_raw_layers.rds")

# On layers: layers = -2 (the default) means "second from the top". Negative
# indexing counts back from the last layer, so with 12 layers -2 is layer 11.
# Layer 0 is the input embedding, not a hidden state - normally not used.
# The upper-middle layers are the usual choice; the very top layers are
# specialised for the model's own training task.

# --- FALLBACK -----------------------------------------------------------------
# If the model download is still running, the package ships one of these objects
# ready-made (bert-base-uncased, layers 11 and 12, but only 6 dimensions wide -
# right shape, toy numbers). It feeds straight into B3:
#
#   raw_layers <- raw_embeddings_1
#   raw_layers$context_tokens$harmonywords[[1]]
# ------------------------------------------------------------------------------


In [ ]:
# ------------------------------------------------------------------------------
# B3. Aggregate: from many numbers per token to one vector per text
# ------------------------------------------------------------------------------
# Step 2 of 3. textEmbedLayerAggregation() collapses that stack twice over:
#
#   aggregation_from_layers_to_tokens  layers   -> one vector per token
#   aggregation_from_tokens_to_texts   tokens   -> one vector per text
#
# Both accept "mean", "min", "max" or "concatenate". Concatenating two 768-wide
# layers gives 1536 columns and keeps everything; averaging them gives 768 and
# throws information away. Neither is right - but whichever you pick, it is a
# reported analytic decision, not a detail.
#
# Note you pass the $context_tokens element, not the whole object.

sentence_embedding <- textEmbedLayerAggregation(
  raw_layers$context_tokens,
  layers                            = 11:12,
  aggregation_from_layers_to_tokens = "concatenate",
  aggregation_from_tokens_to_texts  = "mean"
)

sentence_embedding$texts
dim(sentence_embedding$texts)    # 1 row, id + 2 x 768 dimensions

saveRDS(sentence_embedding, "tutorial_output/day1_sentence_embedding.rds")

# Compare with averaging the layers instead - same input, narrower output:
sentence_embedding_mean <- textEmbedLayerAggregation(
  raw_layers$context_tokens,
  layers                            = 11:12,
  aggregation_from_layers_to_tokens = "mean",
  aggregation_from_tokens_to_texts  = "mean"
)

dim(sentence_embedding_mean$texts)

# And to keep the tokens separate instead of averaging them into one text
# vector, set aggregation_from_tokens_to_texts = NULL:
#
# tokens_kept <- textEmbedLayerAggregation(
#   raw_layers$context_tokens,
#   layers                           = 11:12,
#   aggregation_from_tokens_to_texts = NULL,
#   return_tokens                    = TRUE
# )


In [ ]:
# ------------------------------------------------------------------------------
# B4. Name the dimensions
# ------------------------------------------------------------------------------
# Step 3 of 3, and the least glamorous one. Once you embed more than one
# variable, bare Dim1 ... Dim768 collide the moment you put two embeddings side
# by side. textDimName() adds or strips the variable-name suffix.

names(sentence_embedding$texts)[1:5]                       # id, Dim1, Dim2 ...

named <- textDimName(sentence_embedding$texts,
                     dim_names = TRUE, name = "sentence")
names(named)[1:5]                                          # Dim1_sentence ...

back <- textDimName(named, dim_names = FALSE)
names(back)[1:5]

# textEmbed() calls exactly this for you at the end - which is why its output
# columns are called Dim1_harmonywords and not Dim1.
#
# Careful: it renames EVERY column, including id. Do the naming on the embedding
# columns, not on an object you still need the id from.


In [ ]:
# ------------------------------------------------------------------------------
# B5. Does it behave sensibly? Cosine similarity
# ------------------------------------------------------------------------------
# An embedding is only useful if texts that mean similar things end up close
# together. Cosine similarity is the standard way to ask: the cosine of the
# angle between two vectors, running from -1 to 1, insensitive to length.
#
# Here we use textEmbed(), which is B2 + B3 + B4 in one call - now that you know
# what it is doing.

sentences <- c(
  "I feel calm and balanced",     # 1
  "My life is in harmony",        # 2  - close in meaning to 1
  "The printer is out of paper"   # 3  - not close to anything
)

s_emb <- textEmbed(sentences, model = "bert-base-uncased")

s_emb$texts$texts
dim(s_emb$texts$texts)

saveRDS(s_emb, "tutorial_output/day1_sentence_embeddings_3.rds")

# textSimilarity() compares ROW BY ROW, so the two inputs must have the same
# number of rows. Give it one row against another.

textSimilarity(s_emb$texts$texts[1, ], s_emb$texts$texts[2, ])   # calm vs harmony
textSimilarity(s_emb$texts$texts[1, ], s_emb$texts$texts[3, ])   # calm vs printer

# The first should be clearly higher than the second. If it is not, something is
# wrong with the model or the initialisation - say so before we continue.
#
# It also works on whole columns at once, one comparison per row - which is how
# you would score every participant against a norm text:
#
# textSimilarity(embeddings_a$texts$x, embeddings_b$texts$y)
#
# Related functions, same input shape:
#   textDistance()          euclidean distance instead of cosine similarity
#   textSimilarityNorm()    compare many texts against ONE reference embedding
#   textSimilarityMatrix()  every text against every other text
#
# One caution before you over-read a single number. Cosine similarity is high
# for any two fluent English sentences - the floor is not 0. What is meaningful
# is the DIFFERENCE between similarities, not their absolute size.


In [ ]:
# ------------------------------------------------------------------------------
# B6. What you now have
# ------------------------------------------------------------------------------
# A word embedding is a wide numeric data frame: one row per text, one column
# per dimension. Nothing about it is special to R or to this package - it is
# just a matrix of predictors, produced by three decisions you now control:
# which model, which layers, and how to aggregate.
#
# Day 2 takes that object and does statistics with it: trains a model to
# predict a rating scale, and applies models other people have already trained.

list.files("tutorial_output")   # everything today produced

# Next: day2_all.R
# ==============================================================================


# Day 2 · From numbers to scores

Everything runs on `Language_based_assessment_data_8`, the example dataset that ships inside the text package - this day is standalone. **PART A** embeds real open-ended responses; **PART B** trains your own language-based assessment model; **PART C** applies an existing model from the L-BAM Library; **PART D**: how much data do you need, and what may you claim.

Train before you apply: once you have built a model yourself, it is obvious what an L-BAM model *is*, and what you are trusting when you apply someone else's.


In [ ]:
library(text)   # already initialized to the shared environment by the setup cell

dir.create("tutorial_output", showWarnings = FALSE)


## PART A  ·  From responses to embeddings

In [ ]:
# ------------------------------------------------------------------------------
# A1. The example data that ships with text
# ------------------------------------------------------------------------------
# Language_based_assessment_data_8 is a 40 x 8 tibble: open-ended responses
# alongside validated rating-scale scores - exactly the structure you need to
# show that language predicts an outcome.

Language_based_assessment_data_8

dplyr::glimpse(Language_based_assessment_data_8)

# The eight columns:
#   harmonywords       words describing harmony in life
#   satisfactionwords  words describing satisfaction with life
#   harmonytexts       a longer written description of harmony
#   satisfactiontexts  a longer written description of satisfaction
#   hilstotal          Harmony In Life Scale score        (the outcome we use)
#   swlstotal          Satisfaction With Life Scale score
#   age, gender        demographics
#
# Index by name, never by position - the column order in the object is not the
# order given in the help file.

# Today's pair:
#   harmonywords  the language (predictor)
#   hilstotal     the rating scale (outcome)

Language_based_assessment_data_8$harmonywords[1:5]
summary(Language_based_assessment_data_8$hilstotal)

# A look at the language itself, before modelling anything
textDescriptives(Language_based_assessment_data_8$harmonywords)


In [ ]:
# ------------------------------------------------------------------------------
# A2. Embed
# ------------------------------------------------------------------------------
# Yesterday we did this in three explicit steps - raw layers, aggregation,
# dimension names. textEmbed() is those three steps in one call.
#
# aggregation_from_tokens_to_word_types = "mean" additionally returns one
# embedding per unique word type. We do not need that to predict, but Day 3's
# word plots do - so ask for it now and save it. Without this argument
# $word_types is absent.
#
# Note the argument is model = (singular). The first run downloads the model:
# a few hundred MB and some patience.

h_embeddings <- textEmbed(
  Language_based_assessment_data_8["harmonywords"],
  model                                 = "mixedbread-ai/mxbai-embed-large-v1",
  aggregation_from_tokens_to_word_types = "mean"
)

names(h_embeddings)                      # tokens, texts, word_types

h_embeddings$texts$harmonywords          # one row per participant
dim(h_embeddings$texts$harmonywords)     # participants x dimensions

h_embeddings$word_types$harmonywords     # one row per unique word type

saveRDS(h_embeddings, "tutorial_output/h_embeddings.rds")


# --- FALLBACK -----------------------------------------------------------------
# Embedding is the first slow step of the day. In order of preference:
#
# 1. The pre-computed object - identical to the above:
#      h_embeddings <- readRDS("tutorial_output/h_embeddings_precomputed.rds")
#
# 2. The ready-made embeddings that ship with the package. Same 40 people,
#    but only 10 dimensions - instant, and fine for following the code:
#      h_embeddings <- word_embeddings_4
#      h_embeddings$texts$harmonywords     # 40 x 10
#
#    Everything below works with either object. Accuracy from the 10-dimension
#    version will be lower; that is the dimensions, not your code.
# ------------------------------------------------------------------------------


In [ ]:
# ##############################################################################
# PART B  ·  Training your own model
# ##############################################################################
#
# What "training" means here: ordinary regularised regression, with embedding
# dimensions as predictors and your rating scale as the outcome. The transformer
# did the hard part already. This step is statistics you already know.


In [ ]:
# ------------------------------------------------------------------------------
# B1. Train
# ------------------------------------------------------------------------------
# textTrainRegression() cross-validates internally: it repeatedly holds part of
# the data out, fits on the rest, and predicts the held-out part. The accuracy
# it reports is therefore out-of-sample, which is the only kind worth reporting.
#
# x = the embeddings, y = the outcome you want to predict.

trained_hils <- textTrainRegression(
  x = h_embeddings$texts$harmonywords,
  y = Language_based_assessment_data_8$hilstotal
)

trained_hils

saveRDS(trained_hils, "tutorial_output/trained_hils.rds")

# --- FALLBACK -----------------------------------------------------------------
# Training is the slowest step of Day 2. If it is still going, load this:
#   trained_hils <- readRDS("tutorial_output/trained_hils_precomputed.rds")
# ------------------------------------------------------------------------------


In [ ]:
# ------------------------------------------------------------------------------
# B2. Read the output
# ------------------------------------------------------------------------------
# The object carries the cross-validated predictions and their correlation with
# the observed outcome. That correlation IS the model's accuracy.

trained_hils$results
trained_hils$predictions          # one predicted value per person
names(trained_hils$predictions)   # check the column names before using them

# How good is good? Two reference points, not one:
#
#   The rating scale's own reliability sets the ceiling. A model cannot
#   correlate with a measure more strongly than the measure correlates with
#   itself. Compare your r to sqrt(alpha), not to 1.0.
#
#   The best published models for well-being constructs reach the region of
#   r = .6-.85 (Kjell et al., 2022). Well below that on similar data is a signal
#   to look at your sample size, your prompt, or your embedding model.
#
# And remember what N is here: 40 people. This is a demonstration of the
# pipeline, not an estimate you would publish.

# Always plot as well as read the number - a correlation can hide a badly
# non-linear or floor/ceiling-bound relationship.
plot(
  Language_based_assessment_data_8$hilstotal,
  trained_hils$predictions$predictions,
  xlab = "Harmony In Life Scale (self-report)",
  ylab = "Predicted from language (cross-validated)",
  pch  = 16,
  col  = grDevices::adjustcolor("black", alpha.f = 0.4)
)


In [ ]:
# ------------------------------------------------------------------------------
# B3. What you just built
# ------------------------------------------------------------------------------
# trained_hils is a model object: the weights that turn an embedding into a
# predicted HILS score. Point it at new text from a comparable population and it
# produces scores without anyone filling in a questionnaire.
#
# That object is exactly what an L-BAM model is. PART C applies one that
# somebody else trained - which is the same thing, minus the part where you know
# the sample it came from.


In [ ]:
# ##############################################################################
# PART C  ·  Applying a pretrained model from the L-BAM Library
# ##############################################################################
#
# The L-BAM (Language-Based Assessment Model) Library is a set of openly shared,
# already-trained models. Instead of training your own on your own labelled
# data, you apply someone else's model to your text and get a score.
#
# This is the fastest route from language to a usable measure - and the one that
# needs the most care, because a model trained elsewhere may not transfer here.
#
# Library:   https://r-text.org/articles/LBAM.html
# Reference: Nilsson et al. (2026), Advances in Methods and Practices in
#            Psychological Science


In [ ]:
# ------------------------------------------------------------------------------
# C1. Browse the library
# ------------------------------------------------------------------------------
# textLBAM() returns the catalogue: what each model assesses, what it was
# trained on, and its reported accuracy.

lbam <- textLBAM()

dim(lbam)
names(lbam)

# Look before you pick. The columns that decide whether a model fits YOUR study:
lbam[, c("Name", "Construct_Concept_Behaviours", "Language",
         "Participants_training", "Model_Type")]

# View(lbam)   # in RStudio, much easier to read

# There are two similarly named columns and they mean different things:
#   Model_type  the statistical model  (e.g. "ridge regression")
#   Model_Type  how it is applied      (e.g. "text-trained", "implicit-motives")


In [ ]:
# ------------------------------------------------------------------------------
# C2. Apply a model
# ------------------------------------------------------------------------------
# textAssess() takes raw texts and a model name and returns predicted scores.
# It embeds internally, so you pass texts, not embeddings.
#
# dim_names = FALSE matters: several L-BAM models store their predictors as
# bare Dim1 ... DimN, and with dim_names = TRUE the column naming breaks.
#
# (textAssess, textPredict and textClassify are three names for one function.)

predicted_hils <- textAssess(
  model_info = "harmony_text_roberta23_kjell2022",
  texts      = Language_based_assessment_data_8$harmonytexts,
  dim_names  = FALSE
)

predicted_hils

# The prediction column is named after the model's training outcome, so it
# differs from model to model and can contain awkward characters. Never hard-code
# it - look it up, or take the first column.
names(predicted_hils)

hils_pred <- predicted_hils[[1]]

saveRDS(predicted_hils, "tutorial_output/predicted_hils.rds")

# --- FALLBACK -----------------------------------------------------------------
# predicted_hils <- readRDS("tutorial_output/predicted_hils_precomputed.rds")
# hils_pred      <- predicted_hils[[1]]
# ------------------------------------------------------------------------------


In [ ]:
# ------------------------------------------------------------------------------
# C3. Validate in YOUR data - never skip this
# ------------------------------------------------------------------------------
# A published accuracy figure describes the model's original validation sample.
# It is not a promise about your sample. If you have a rating scale alongside
# your text, check the correlation yourself. This is the step PART B taught you
# to expect: you cross-validated your own model, so ask the same of theirs.

cor.test(hils_pred, Language_based_assessment_data_8$hilstotal)

# Read the result as a transfer check:
#   close to the published figure  -> the model transfers to your data
#   clearly lower                  -> it does not; report that, and train your
#                                     own instead (PART B)
#   near zero                      -> stop. Do not report scores from this model.

plot(
  Language_based_assessment_data_8$hilstotal,
  hils_pred,
  xlab = "Harmony In Life Scale (self-report)",
  ylab = "Predicted from language (L-BAM)",
  pch  = 16,
  col  = grDevices::adjustcolor("black", alpha.f = 0.4)
)
abline(lm(hils_pred ~ Language_based_assessment_data_8$hilstotal))


## PART D  ·  How much data, and what you may claim

In [ ]:
# ------------------------------------------------------------------------------
# D1. The learning curve
# ------------------------------------------------------------------------------
# The most common question in this workshop. Rather than guessing, plot accuracy
# as a function of training-set size.
#
# If the curve is still climbing at your full N, more participants would help.
# If it has flattened, collecting more of the same data will not.
#
# This refits the model many times - slow. Run it during the break, or load the
# pre-computed version.

# trained_n <- textTrainN(
#   x               = h_embeddings$texts$harmonywords,
#   y               = Language_based_assessment_data_8$hilstotal,
#   sample_percents = c(25, 50, 75, 100),
#   n_cross_val     = 1
# )
#
# textTrainNPlot(trained_n)
#
# FALLBACK: trained_n <- readRDS("tutorial_output/trained_n_precomputed.rds")


In [ ]:
# ------------------------------------------------------------------------------
# D2. Sharing the model - the open-science step
# ------------------------------------------------------------------------------
# A trained model that stays on your laptop cannot be independently validated,
# which means nobody can check whether it works outside your sample. The L-BAM
# Library exists so that models can be shared and tested by others - and PART C
# only worked because someone did this.
#
# What a shareable model needs documented alongside it:
#   - the exact embedding model used (name and version)
#   - the prompt participants answered, verbatim
#   - the population: N, age, language, recruitment
#   - the outcome measure and its reliability in your sample
#   - the cross-validated accuracy, and how you cross-validated
#   - known limits: where you expect it NOT to transfer
#
# Submission process: https://r-text.org/articles/LBAM.html
#
# The model itself contains no participant text - sharing it does not share your
# data. That is what makes this practical for sensitive material.


In [ ]:
# ------------------------------------------------------------------------------
# D3. What a predicted score is, and is not
# ------------------------------------------------------------------------------
# A predicted score is an estimate of how someone would have answered a
# questionnaire. That is a real, useful thing. It is not:
#
#   - a diagnosis
#   - a measure of one person at one moment that you can act on clinically
#   - valid outside the population, prompt, and language it was validated in
#
# Report, every time: the exact model name, its source, the population it was
# trained on, and the accuracy YOU observed in YOUR data.

# citation("text")
# sessionInfo()

list.files("tutorial_output")   # everything today produced

# Next: day3_all.R
# ==============================================================================


# Day 3 · From scores to understanding - topics, word plots, publication

Everything runs on the example data that ships inside the packages (`dep_wor_data` from topics), so this day is standalone. **PART A**: which exact expressions (n-grams) relate to the outcome. **PART B**: the same question asked of *themes* (topics). **PART C**: how to read these plots - and how not to. **PART D**: publication-resolution export and what to report.

topics uses **Java**, not Python - the setup cell configured Java (and set its memory limit) before starting it, so there is nothing to restart here.


In [ ]:
# The Java heap size (java.parameters) must be set before the JVM starts;
# the setup cell already did that (options(java.parameters = "-Xmx5000m")).

library(topics)
library(dplyr)

dir.create("tutorial_output", showWarnings = FALSE)

# rJava and mallet are Suggests of topics, so they are not installed with it
# automatically; PART B needs them (PART A does not). stopwords is used in A3.
Sys.which("java")   # must not be empty
if (!requireNamespace("mallet",    quietly = TRUE)) install.packages(c("rJava", "mallet"))
if (!requireNamespace("stopwords", quietly = TRUE)) install.packages("stopwords")


## PART A  ·  n-grams - which exact expressions relate to the outcome

In [ ]:
# ------------------------------------------------------------------------------
# A1. The example data that ships with topics
# ------------------------------------------------------------------------------
# dep_wor_data: 500 participants, 18 variables. Each person gave open-ended
# responses about their depression AND about their worry, in three formats
# (single words, phrases, free text), alongside validated scales and health
# outcomes.

dep_wor_data

glimpse(dep_wor_data)

# The columns we use today:
#   Depphrase / Worphrase   the open-ended responses (phrase format)
#   PHQ9tot                 depression severity  (the outcome)
#   GAD7tot                 anxiety severity
#   Age, Gender             controls  (Gender: 0 = male, 1 = female)

dep_wor_data$Depphrase[1:3]
summary(dep_wor_data$PHQ9tot)


In [ ]:
# ------------------------------------------------------------------------------
# A2. Reshape to long format
# ------------------------------------------------------------------------------
# We stack the depression responses and the worry responses into one Language
# column, with an indicator for which prompt each row came from. That lets us
# ask two different questions of the same object:
#
#   does the language differ BETWEEN prompts?      (DepWor as the variable)
#   does it track symptom severity WITHIN people?  (PHQ as the variable)

Language <- c(dep_wor_data$Depphrase, dep_wor_data$Worphrase)

# 0 = depression language, 1 = worry language
DepWor <- c(
  rep(0, length(dep_wor_data$Depphrase)),
  rep(1, length(dep_wor_data$Worphrase))
)

# Each person appears twice, so their scores are repeated too
PHQ    <- rep(dep_wor_data$PHQ9tot, 2)
Age    <- rep(dep_wor_data$Age,     2)
Gender <- rep(dep_wor_data$Gender,  2)

dep_wor_language <- tibble(Language, DepWor, PHQ, Age, Gender)

dep_wor_language

# Note the dependency this creates: rows are not independent, because each
# person contributes two. Say so in your Methods section.


In [ ]:
# ------------------------------------------------------------------------------
# A3. Structure the language into n-grams
# ------------------------------------------------------------------------------
# An n-gram is a run of n consecutive words. ngram_window = c(1, 3) keeps single
# words, pairs and triples - so "not" and "not sleeping" are separate features.
# That matters: single words alone lose negation and common phrases.

ngrams <- topicsGrams(
  data           = dep_wor_language$Language,
  ngram_window   = c(1, 3),
  stopwords      = stopwords::stopwords("en", source = "snowball"),
  occurance_rate = 0.01,   # keep n-grams appearing in >= 1% of responses
  pmi_threshold  = 3       # keep multi-word n-grams that hang together
)

ngrams

# Three decisions to make deliberately, not by default:
#
#   stopwords       Removing "the", "and", "is" focuses on content. But for
#                   clinical language function words can carry the signal -
#                   first-person pronouns predict depression in several
#                   literatures. Consider running it both ways.
#
#   occurance_rate  Too low and you test thousands of rare n-grams, paying a
#                   heavy multiple-comparison price. Too high and you only see
#                   the obvious. 0.01 is a starting point, not a rule.
#
#   pmi_threshold   Pointwise mutual information. Keeps phrases whose words
#                   genuinely go together and drops incidental co-occurrences.
#                   It belongs HERE, at the n-gram step - topicsTest() has no
#                   such argument and will error if you pass it one.

saveRDS(ngrams, "tutorial_output/ngrams.rds")
# FALLBACK: ngrams <- readRDS("tutorial_output/ngrams_precomputed.rds")


In [ ]:
# ------------------------------------------------------------------------------
# A4. Test which n-grams relate to the outcome
# ------------------------------------------------------------------------------
# One regression per n-gram, predicting PHQ severity, controlling for age and
# gender, with p-values corrected for multiple testing.

ngrams_test <- topicsTest(
  data            = dep_wor_language,
  ngrams          = ngrams,
  test_method     = "linear_regression",
  x_variable      = "PHQ",
  controls        = c("Age", "Gender"),
  p_adjust_method = "fdr"
)

ngrams_test

# What the arguments are doing:
#
#   controls         age and gender are partialled out, so a surviving n-gram
#                    is not simply a demographic marker
#   p_adjust_method  "fdr" controls the false discovery rate. With hundreds of
#                    n-grams tested, uncorrected p-values are meaningless.
#                    Never report these plots without correction.

saveRDS(ngrams_test, "tutorial_output/ngrams_test.rds")
# FALLBACK: ngrams_test <- readRDS("tutorial_output/ngrams_test_precomputed.rds")


In [ ]:
# ------------------------------------------------------------------------------
# A5. The n-gram plot
# ------------------------------------------------------------------------------
# Two word clouds: expressions associated with HIGHER PHQ, and with LOWER PHQ.
#
# color_scheme takes four colours here, as two low-to-high gradient pairs:
#   [1] [2]  the negative-association cloud, faint -> strong
#   [3] [4]  the positive-association cloud, faint -> strong
# (These four are also the package default, spelled out so you can change them.)
#
# Careful: topicsTest() defaults to p_adjust_method = "fdr" but topicsPlot()
# defaults to "none". Set it explicitly in both, or the figure will show more
# than the test supported.

ngram_plots <- topicsPlot(
  ngrams          = ngrams,
  test            = ngrams_test,
  ngrams_max      = 30,
  p_alpha         = 0.05,
  p_adjust_method = "fdr",
  color_scheme    = c(
    "#d0d0d0", "#ff7c00",   # low PHQ  (orange)
    "#d0d0d0", "#00508c"    # high PHQ (blue)
  )
)

names(ngram_plots)   # positive_association, negative_association, overview_plot

ngram_plots$positive_association
ngram_plots$negative_association
ngram_plots$overview_plot

saveRDS(ngram_plots, "tutorial_output/ngram_plots.rds")

# If nothing survives correction you will see "No significant terms. No
# wordclouds generated." That is a result, not a bug. To show the mechanics
# anyway, rerun the call above with p_alpha = 1 - and say out loud that you
# have done so.


In [ ]:
# ##############################################################################
# PART B  ·  topics - the same question asked of themes
# ##############################################################################
#
# n-grams test individual expressions. Topics group co-occurring words into
# themes first, then test the themes. Fewer tests, more interpretable units,
# but a layer of modelling between you and the data.
#
# Use n-grams when you want to know which exact words matter; topics when you
# want themes. Both, if you want to show the result is not an artefact of one
# choice.
#
# This part needs Java + rJava + mallet. If it fails, PART A already gave you a
# complete result and PART C applies to it unchanged.


In [ ]:
# ------------------------------------------------------------------------------
# B1. Document-term matrix
# ------------------------------------------------------------------------------

dtm <- topicsDtm(
  data           = dep_wor_language$Language,
  ngram_window   = c(1, 3),
  stopwords      = stopwords::stopwords("en", source = "snowball"),
  removal_mode   = "frequency",
  removal_rate_least = 4,
  removal_rate_most  = 500
)

# Check the vocabulary before modelling it - this is where you catch a stopword
# list that removed too much, or a corpus dominated by one word.
dtm_evaluation <- topicsDtmEval(dtm)
dtm_evaluation$frequency_plot

saveRDS(dtm, "tutorial_output/dtm.rds")


In [ ]:
# ------------------------------------------------------------------------------
# B2. Fit the topic model
# ------------------------------------------------------------------------------
# num_topics is your choice and it changes the answer. Fit a few values and
# report that you did.

model <- topicsModel(
  dtm            = dtm,
  num_topics     = 20,
  num_top_words  = 10,
  num_iterations = 1000,
  seed           = 42
)

names(model)       # what the fitted object contains
model$summary      # the topics, with their top words

saveRDS(model, "tutorial_output/topics_model.rds")
# FALLBACK: model <- readRDS("tutorial_output/topics_model_precomputed.rds")


In [ ]:
# ------------------------------------------------------------------------------
# B3. Topic prevalence per response, then test
# ------------------------------------------------------------------------------

preds <- topicsPreds(
  model = model,
  data  = dep_wor_language$Language
)

topics_test <- topicsTest(
  data            = dep_wor_language,
  model           = model,
  preds           = preds,
  x_variable      = "PHQ",
  controls        = c("Age", "Gender"),
  test_method     = "linear_regression",
  p_adjust_method = "fdr"
)

topics_test

saveRDS(preds,       "tutorial_output/topics_preds.rds")
saveRDS(topics_test, "tutorial_output/topics_test.rds")


In [ ]:
# ------------------------------------------------------------------------------
# B4. Plot the topics
# ------------------------------------------------------------------------------
# With one x_variable you get three colour categories, and the returned list is
# named for them:
#   square1  significantly NEGATIVE association
#   square2  not significant
#   square3  significantly POSITIVE association
#
# Each square is itself a list of one plot per topic, named t_<topic number>.

topic_plots <- topicsPlot(
  model           = model,
  test            = topics_test,
  p_alpha         = 0.05,
  p_adjust_method = "fdr"
)

names(topic_plots)

topic_plots$square3          # every topic linked to higher PHQ
# names(topic_plots$square3) # which topic numbers those are
# topic_plots$square3[[1]]   # one topic on its own

topic_plots$legend
topic_plots$overview_plot    # a patchwork composition of the top topics

saveRDS(topic_plots, "tutorial_output/topic_plots.rds")

# If you want your own colours here the vector must be SIX long, interleaved as
# background/front pairs for the three categories:
#   c("#d0d0d0","#ff7c00",  "#d0d0d0","#a4a4a4",  "#d0d0d0","#00508c")
# Two variables (x_variable + y_variable) makes it a 3x3 grid and eighteen.


In [ ]:
# ##############################################################################
# PART C  ·  How to read these plots - and how NOT to
# ##############################################################################
#
# Reference: Eijsbroek et al. (2026). Multiple Methods for Visualizing Human
#            Language: A Tutorial for Social and Behavioural Scientists. AMPPS.
#
# Two visual channels, two different meanings. Confusing them is the single most
# common error:
#
#   SIZE      = frequency. How often the expression was used.
#   COLOUR    = the tested association, and its strength.
#   POSITION  = chosen for legibility, not meaning. Nothing is encoded in x/y
#               within a word cloud. Do not read clusters into it.
#
# So a large faint word is COMMON but WEAKLY related to the outcome. A small
# strong-coloured word is RARE but RELIABLY associated. Readers routinely read
# the biggest word as the main finding. Say the sentence above out loud when you
# present the figure.
#
# Four claims these plots do NOT support:
#
#   "People who say X are depressed."
#       It is an association across a sample, not a statement about a person.
#
#   "X causes / indicates depression."
#       Cross-sectional correlational data. Nothing here is causal.
#
#   "X is absent from the other group."
#       Not significant is not the same as not present.
#
#   "This generalises."
#       It describes this sample, this prompt, this language. n-grams are more
#       sample-specific than embeddings - they will not transfer as readily.
#
# And a figure without a legend is not interpretable on its own. topicsPlot()
# returns one; use it.


In [ ]:
# ------------------------------------------------------------------------------
# C1. The other route to a word plot - projection in embedding space
# ------------------------------------------------------------------------------
# textProjection() takes a different path to a similar-looking figure. Instead
# of counting n-grams, it places each word type in the embedding space and
# projects it onto the outcome dimension.
#
# It answers a related but distinct question, so the two figures will not match
# exactly - and that is informative, not a problem.
#
# The text package ships a finished projection object, so you can see the plot
# without waiting for embeddings. Note this switches to the text package's
# Python environment - if you also want to run topics afterwards, restart R.

# library(text)
#
# DP_projections_HILS_SWLS_100$word_data
#
# textProjectionPlot(
#   word_data           = DP_projections_HILS_SWLS_100$word_data,
#   min_freq_words_test = 1,
#   plot_n_word_extreme = 10,
#   p_adjust_method     = "fdr",
#   title_top           = "Harmony in life words projected on HILS",
#   x_axes_label        = "Low HILS   ...   High HILS"
# )
#
# To build one from your own data instead (Day 2 produced h_embeddings):
#
# h_embeddings <- readRDS("tutorial_output/h_embeddings.rds")   # saved on Day 2
# projection_data <- textProjection(
#   words                 = Language_based_assessment_data_8$harmonywords,
#   word_embeddings       = h_embeddings$texts$harmonywords,
#   word_types_embeddings = h_embeddings$word_types$harmonywords,
#   x                     = Language_based_assessment_data_8$hilstotal
# )
# textProjectionPlot(projection_data$word_data)
#
# A third view - semantic centrality, which words are most typical of the
# response set rather than most related to an outcome. Ready-made example data:
#
# textCentralityPlot(word_data = centrality_data_harmony)


## PART D  ·  Export, and what goes in the manuscript

In [ ]:
# ------------------------------------------------------------------------------
# D1. Publication-resolution figures
# ------------------------------------------------------------------------------
# Journals want 300+ dpi, and vector where they accept it. A screenshot of the
# RStudio plot pane will be rejected.
#
# Two ways to save. Either let topicsPlot() do it - give it save_dir and files
# land in <save_dir>/seed_<seed>/wordclouds/ - or save the returned objects
# yourself with ggsave(), which gives you control over size and dpi.

# Automatic:
# topicsPlot(
#   ngrams        = ngrams,
#   test          = ngrams_test,
#   save_dir      = "tutorial_output",
#   figure_format = "png",     # note: "png", not ".png"
#   width         = 8,
#   height        = 6
# )

# Manual - raster, for journals that want TIFF/PNG
ggplot2::ggsave(
  filename = "tutorial_output/ngram_plot_phq_high.png",
  plot     = ngram_plots$positive_association,
  width    = 180,          # mm - check your journal's column width
  height   = 140,
  units    = "mm",
  dpi      = 600,
  bg       = "white"       # without this you get a transparent background
)

# Manual - vector, which stays sharp at any size
ggplot2::ggsave(
  filename = "tutorial_output/ngram_plot_phq_high.pdf",
  plot     = ngram_plots$positive_association,
  width    = 180,
  height   = 140,
  units    = "mm"
)

# The overview panel is a patchwork composition; ggsave handles it too.
ggplot2::ggsave(
  filename = "tutorial_output/ngram_overview.pdf",
  plot     = ngram_plots$overview_plot,
  width    = 260,
  height   = 160,
  units    = "mm"
)

# Check the saved file, not the plot pane - text that looks fine on screen is
# often unreadably small once the figure is scaled to a journal column.


In [ ]:
# ------------------------------------------------------------------------------
# D2. What to put in the manuscript
# ------------------------------------------------------------------------------
# Methods must state: the n-gram or topic settings (ngram_window, stopword list,
# occurance_rate, pmi_threshold, num_topics), the test method, every control
# variable, the multiple-comparison correction, and the package versions.
#
# The caption must state what size and colour encode. Assume the reader will
# look only at the figure.

# citation("topics")
# citation("text")
# citation("talk")
# sessionInfo()


In [ ]:
# ------------------------------------------------------------------------------
# D3. What is now in tutorial_output/
# ------------------------------------------------------------------------------
# Across the three days, everything slow was saved rather than left in memory:
#
#   Day 1   day1_transcription, day1_audio_embedding, day1_transcript,
#           day1_segment_embeddings, day1_raw_layers, day1_sentence_embedding,
#           day1_sentence_embeddings_3
#   Day 2   h_embeddings, trained_hils, predicted_hils
#   Day 3   ngrams, ngrams_test, dtm, topics_model, topics_preds, topics_test,
#           ngram_plots, topic_plots, and the exported figures
#
# This is the habit worth taking home. Embedding and training are the expensive
# steps; saving them means a re-run of your analysis is seconds, not hours, and
# it means the object a reviewer asks about still exists six months later.
#
# What NOT to put here: the participant texts themselves, if they are sensitive.
# Fitted models and embeddings are derived and shareable; raw responses are not.

list.files("tutorial_output")

# ==============================================================================
